# Setup

In [2]:
# @title Install Packages
%%capture
!pip install mapclassify
!pip install SPARQLWrapper
!pip install rdflib

UsageError: Line magic function `%%capture` not found.


In [ ]:
# @title Import packages and define functions
from branca.element import Figure                                   # For controlling the size of the final map
import folium                                                       # For map layer control
from folium.plugins import StripePattern                            # For filling polygons with a striped pattern
import geopandas as gpd                                             # For geospatial dataframes
import pandas as pd                                                 # For dataframes
from shapely import wkt                                             # For working with WKT coordinates in a GeoDataFrame
from shapely.ops import transform                                   # To fix well coordinates (swap from (y, x) to (x, y)) - can be removed in the future once the data are fixed in FRINK
from SPARQLWrapper import SPARQLWrapper2, JSON, GET, POST, DIGEST   # For querying SPARQL endpoints
import rdflib                                                       # For working with URIs
from datetime import date                                           # For adding the date to saved map files

def convertToDataframe(results):
  d = []
  for x in results.bindings:
        row = {}
        for k in x:
            v = x[k]
            vv = rdflib.term.Literal(v.value, datatype=v.datatype).toPython()  # type: ignore[no-untyped-call]
            row[k] = vv
        d.append(row)
  df = pd.DataFrame(d)
  return df

# Question: What water wells in Maine (or a Maine county) are potentially hydrologically connected to PFAS contaminated private water supply wells where results have been at least *x* ng/L?

In [1]:
# @title Select administrative region
admin_region = "23005 (Cumberland County, Maine)" # @param [ "23 (Maine)", "23001 (Androscoggin County, Maine)", "23005 (Cumberland County, Maine)", "23011 (Kennebec County, Maine)", "23019 (Penboscot County, Maine)", "23031 (York County, Maine)" ] {"allow-input":true}

regionCode = admin_region.split()[0]
if regionCode == 'All':
  regionFilter = ''
else:
  if len(regionCode) <= 5:
    regionURI = 'kwgr:administrativeRegion.USA.' + regionCode
    print(regionURI)
    regionFilter = '''?county rdf:type kwg-ont:AdministrativeRegion_3 ;
            kwg-ont:administrativePartOf+ ''' + regionURI + ''' . '''
  else:
    # need to implement region filters with datacommons geoids.
    regionFilter = ''

kwgr:administrativeRegion.USA.23005


In [ ]:
# @title Select PFAS type
pfas_choice = "PFOA (perfluorooctanoic acid)" # @param ['PFOA (perfluorooctanoic acid)', 'PFOS (perfluorooctane sulfonic acid)', 'PFHxS (perfluorohexane sulfonic acid)', 'PFNA (perfluorononanoic acid)', 'PFDA (perfluorodecanoic acid)', 'PFHxA (hexafluoropropylene oxide dimer acid (GenX))'] {"allow-input":true}
pfas_type = f'{pfas_choice.split()[0]}_A'
print(pfas_type)

PFOA_A


In [ ]:
# @title Select minimum contamination level
min_contamination = 50 # @param {"allow-input":true}
print(min_contamination)

50


# Query

In [ ]:
# @title Retrieve aquifers, s2 cells, and sample results
me_egad_sample_point_type = "PWSW" # private water supply well
unit = 'NanoGM-PER-L'
query = f"""
PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX gwml2: <http://gwml2.org/def/gwml2#>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me-mgs#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT DISTINCT *
WHERE {{
    {regionFilter}
    ?samples2 rdf:type kwg-ont:S2Cell_Level13 ;
              spatial:connectedTo ?county ;
              spatial:connectedTo ?aquifer .

    ?aquifer rdf:type gwml2:GW_Aquifer ;
             geo:hasGeometry/geo:asWKT ?aquiferwkt .
    ?aqs2 rdf:type kwg-ont:S2Cell_Level13 ;
       	   spatial:connectedTo ?aquifer .
    ?well rdf:type me_mgs:MGS-Well ;
          kwg-ont:sfWithin ?aqs2 ;
          rdfs:label ?welllabel ;
          me_mgs:hasUse ?welluseiri ;
          me_mgs:ofWellType ?welltypeiri ;
          me_mgs:wellDepth/qudt:numericValue ?welldepth ;
          me_mgs:wellOverburden/qudt:numericValue ?welloverburden ;
          geo:hasGeometry/geo:asWKT ?wellwkt .
    BIND(REPLACE(str(?welluseiri), "(.*)\\\\.(.*)", "$2") AS ?welluse)
    BIND(REPLACE(str(?welltypeiri), "(.*)\\\\.(.*)", "$2") AS ?welltype)
    BIND(CONCAT(str(?welldepth), " ft") AS ?welldepthft)
    BIND(CONCAT(str(?welloverburden), " ft") AS ?welloverburdenft)

    ?samplept rdf:type me_egad:EGAD-SamplePoint ;
              me_egad:samplePointType me_egad:featureType.{me_egad_sample_point_type} ;
              kwg-ont:sfWithin ?samples2 ;
              geo:hasGeometry/geo:asWKT ?sampleptwkt .
    ?obs coso:observedAtSamplePoint ?samplept ;
         coso:ofSubstance me_egad_data:parameter.{pfas_type} ;
         coso:ofSubstance ?substance ;
         sosa:hasResult/qudt:quantityValue ?resultqv .
    ?resultqv qudt:numericValue ?resultvalue ;
              qudt:hasUnit unit:{unit} .
    BIND(REPLACE(str(?substance), "(.*)\\\\.(.*)", "$2") AS ?pfastype)
    BIND(CONCAT(str(?resultvalue), " ", "{unit}") AS ?result)
    FILTER(?resultvalue > {min_contamination})
}}
"""

print(query)


PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX gwml2: <http://gwml2.org/def/gwml2#>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me-mgs#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT DISTINCT *
WHERE {
    ?county rdf:type kwg-ont:AdministrativeRegion_3 ;
            kwg-ont:administrativePartOf+ kwgr:administrativeRegion.USA.23005 . 
    ?samples2 rdf:type kwg-ont:S2Cell_Level13 ;
              spatial:connectedTo ?c

In [ ]:
%%time
endpointGET = 'https://frink.apps.renci.org/federation/sparql'

sparqlGET = SPARQLWrapper2(endpointGET)
sparqlGET.setHTTPAuth(DIGEST)
sparqlGET.setMethod(POST)
sparqlGET.setReturnFormat(JSON)

sparqlGET.setQuery(query)
query_result = sparqlGET.query()
result = convertToDataframe(query_result)
result.tail()
if result.empty:
    raise Exception("No results returned for chosen input parameters.")

CPU times: user 2.27 s, sys: 1.71 s, total: 3.98 s
Wall time: 12.2 s


# Prep query results for mapping

In [ ]:
# @title Create thematic dataframes
aquifers = result[['aquifer', 'aquiferwkt']].copy()
aquifers.drop_duplicates(inplace=True)
aquifers['aquiferwkt'] = aquifers['aquiferwkt'].apply(wkt.loads)

wells = result[['well', 'welllabel', 'welluse', 'welltype', 'welldepthft', 'welloverburdenft', 'wellwkt']].copy()
wells.drop_duplicates(inplace=True)
wells['wellwkt'] = wells['wellwkt'].apply(wkt.loads)

samplepts = result[['samplept', 'pfastype', 'result', 'sampleptwkt']].copy()
samplepts.drop_duplicates(inplace=True)
samplepts['sampleptwkt'] = samplepts['sampleptwkt'].apply(wkt.loads)

In [ ]:
# @title Create GeoDataFrames
%%capture
aquifers = gpd.GeoDataFrame(aquifers, geometry='aquiferwkt')
aquifers.set_crs(epsg=4326, inplace=True, allow_override=True)

wells = gpd.GeoDataFrame(wells, geometry='wellwkt')
wells.set_crs(epsg=4326, inplace=True, allow_override=True)
wells['wellwkt'] = wells['wellwkt'].apply(lambda geom: transform(lambda x, y: (y, x), geom))

samplepts = gpd.GeoDataFrame(samplepts, geometry='sampleptwkt')
samplepts.set_crs(epsg=4326, inplace=True, allow_override=True)

In [ ]:
# @title Create map layers
%%capture
samplept_color = 'red'
well_color = 'deepskyblue'
aquifer_color = 'blue'
boundweight = 5

map = folium.Map()
sp = StripePattern(angle=-30, color=aquifer_color, space_color='white', space_opacity=0.75).add_to(map)
aquifers.explore(m=map,
                 color=aquifer_color,
                 style_kwds=dict(weight=0.5*boundweight, style_function=lambda x :{'fillPattern': sp}),
                 popup=['aquifer'],
                 tooltip=False,
                 name=f'<span style="color: {aquifer_color};">Aquifers</span>',
                 show=True)
wells.explore(m=map,
              color=well_color,
              style_kwds=dict(weight=2*boundweight),
              popup=['welllabel', 'welltype', 'welluse', 'welldepthft', 'welloverburdenft', 'well'],
              tooltip=False,
              name=f'<span style="color: {well_color};">Potentially connected water wells</span>',
              show=True)
samplepts.explore(m=map,
                  color=samplept_color,
                  style_kwds=dict(weight=2*boundweight),
                  popup=['pfastype', 'result', 'samplept'],
                  tooltip=False,
                  name=f'<span style="color: {samplept_color};">Private well sample points</span>',
                  show=True)

# Show map

In [ ]:
# @title Display map
print()
print(f"Wells (light blue) potentially hydrologically connected to aquifers (dark blue) that are potentially contaminated at overlapping sample points (red).")
print()
print('Use the checkboxes on the right to turn layers on/off.')
print('Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.')
print()
folium.FitOverlays().add_to(map)
folium.LayerControl(collapsed=False).add_to(map)
fig = Figure(width='90%', height=700)
fig.add_child(map)


Wells (light blue) potentially hydrologically connected to aquifers (dark blue) that are potentially contaminated at overlapping sample points (red).

Use the checkboxes on the right to turn layers on/off.
Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.



In [ ]:
# @title Save .html copy of map
today = date.today()
fig.save(f'UC1-CQ3b,5_Water-wells-connected-PWSW-at-least-{min_contamination}-ng-per-L-{pfas_type}_{regionCode}_{today}.html')

# Looking Further

Once the results of the primary query are available, it is possible to query for additional information related to specific results.

Here you can view tables of average test results (by date and type of PFAS) for some different sample points. Select a sample point from the drop down and then press the 'play' button on the left. Different sample points can have very different sample histories.

In [ ]:
sample_point = 'samplePoint.152643' #@param ['samplePoint.143624', 'samplePoint.147579', 'samplePoint.152643', 'samplePoint.154977']
samples_query = f"""
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT ?samp_pt ?obs_date ?pfastype ?num_val ?unit ?cousub_label
WHERE {{
    ?samp_pt rdf:type me_egad:EGAD-SamplePoint ;
             me_egad:samplePointType me_egad:featureType.PWSW ;
             kwg-ont:sfWithin ?cousub .
    ?obs coso:observedAtSamplePoint ?samp_pt ;
         coso:ofSubstance ?substance ;
         sosa:hasResult ?result ;
         coso:observedTime ?obs_date .
    ?result qudt:quantityValue ?quant_val .
    ?quant_val qudt:numericValue ?num_val ;
               qudt:hasUnit ?unit .
    VALUES ?samp_pt {{ me_egad_data:{sample_point} }}
    BIND(REPLACE(str(?substance), "(.*)\\\\.(.*)", "$2") AS ?pfastype)

    ?cousub rdf:type kwg-ont:AdministrativeRegion_3 ;
            rdfs:label ?cousub_label ;
            kwg-ont:administrativePartOf ?county .
    ?county rdfs:label ?county_label ;
            kwg-ont:administrativePartOf ?state .
    ?state rdfs:label ?state_label .
}}
"""
sparqlGET.setQuery(samples_query)
samples_result = sparqlGET.query()
samples = convertToDataframe(samples_result)
if result.empty:
    raise Exception("No results returned for chosen input parameters.")
print(f'\nMean test results by date and PFAS type for {sample_point}\n')
samples.pivot_table('num_val', index='obs_date', columns='pfastype', aggfunc='mean').fillna('--')


Mean test results by date and PFAS type for samplePoint.152643



/tmp/ipython-input-1747992870.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  samples.pivot_table('num_val', index='obs_date', columns='pfastype', aggfunc='mean').fillna('--')


pfastype,6-2_FTS_A,8-2_FTS_A,N-EtFOSAA_A,N-MeFOSAA_A,PFBA_A,PFBS_A,PFDA_A,PFHPA_A,PFHPS_A,PFHXA_A,...,org/DSSTox/v1/DTXSID5030030,org/DSSTox/v1/DTXSID5062760,org/DSSTox/v1/DTXSID6062599,org/DSSTox/v1/DTXSID6067331,org/DSSTox/v1/DTXSID7040150,org/DSSTox/v1/DTXSID8031863,org/DSSTox/v1/DTXSID8031865,org/DSSTox/v1/DTXSID8047553,org/DSSTox/v1/DTXSID8059920,org/DSSTox/v1/DTXSID8062600
obs_date,,,,,,,,,,,,,,,,,,,,,
2023-07-06T12:53:00,21.1,97.0,115.0,0.662,161.0,4.78,404.0,1300.0,103.0,560.0,...,4.78,115.0,404.0,21.1,42.2,578.0,1960.0,20.4,103.0,5.66
